In [1]:
# ClinicalTrials.gov API Ingestion

import requests
import json
import hashlib
import pandas as pd

from pathlib import Path
from datetime import datetime

In [2]:
# Define and validate project cohort

base_url = "https://clinicaltrials.gov/api/v2/studies"

project_query = (
    "AREA[StudyType]INTERVENTIONAL "
    "AND AREA[LeadSponsorClass]INDUSTRY "
    "AND (AREA[InterventionType]DRUG OR AREA[InterventionType]BIOLOGICAL) "
    "AND AREA[StartDate]RANGE[2015-01-01,MAX]"
)

params = {
    "query.term": project_query,
    "pageSize": 5,
    "format": "json",
    "countTotal": "true"
}

response = requests.get(
    base_url,
    params=params,
    timeout=30
)

response.raise_for_status()

filtered_data = response.json()

print("Status:", response.status_code)
print("Studies returned:", len(filtered_data["studies"]))
print("Total studies matching project scope:", filtered_data.get("totalCount"))

Status: 200
Studies returned: 5
Total studies matching project scope: 46911


In [3]:
# Controlled project extraction - 200 studies

params = {
    "query.term": project_query,
    "pageSize": 50,
    "format": "json"
}

all_studies = []
next_page_token = None
max_records = 200
pages_downloaded = 0

while len(all_studies) < max_records:

    print(f"Requesting page {pages_downloaded + 1}...")

    if next_page_token:
        params["pageToken"] = next_page_token
    else:
        params.pop("pageToken", None)

    response = requests.get(
        base_url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    page_data = response.json()

    all_studies.extend(page_data.get("studies", []))

    pages_downloaded += 1

    print(f"Studies downloaded: {len(all_studies)}")

    next_page_token = page_data.get("nextPageToken")

    if not next_page_token:
        break

all_studies = all_studies[:max_records]

print("Extraction complete")
print("Pages downloaded:", pages_downloaded)
print("Final records:", len(all_studies))

Requesting page 1...
Studies downloaded: 50
Requesting page 2...
Studies downloaded: 100
Requesting page 3...
Studies downloaded: 150
Requesting page 4...
Studies downloaded: 200
Extraction complete
Pages downloaded: 4
Final records: 200


In [4]:
# Save raw development extract

raw_extract = {
    "studies": all_studies
}

raw_extract_path = Path("../data/raw/clinicaltrials_project_dev_200.json")

with open(raw_extract_path, "w", encoding="utf-8") as file:
    json.dump(raw_extract, file, indent=2)

print("Saved:", raw_extract_path)
print("File exists:", raw_extract_path.exists())

Saved: ..\data\raw\clinicaltrials_project_dev_200.json
File exists: True


In [5]:
# Create extraction manifest

import hashlib

# Calculate SHA-256 hash of the raw JSON file
sha256 = hashlib.sha256()

with open(raw_extract_path, "rb") as file:
    for chunk in iter(lambda: file.read(8192), b""):
        sha256.update(chunk)

file_hash = sha256.hexdigest()

manifest = {
    "source": "ClinicalTrials.gov API v2",
    "endpoint": base_url,
    "extract_timestamp": datetime.now().astimezone().isoformat(),
    "query": project_query,
    "page_size": 50,
    "pages_downloaded": pages_downloaded,
    "records_downloaded": len(all_studies),
    "development_limit": max_records,
    "raw_file": raw_extract_path.name,
    "sha256": file_hash
}

manifest

{'source': 'ClinicalTrials.gov API v2',
 'endpoint': 'https://clinicaltrials.gov/api/v2/studies',
 'extract_timestamp': '2026-08-26T23:54:40.652302+01:00',
 'query': 'AREA[StudyType]INTERVENTIONAL AND AREA[LeadSponsorClass]INDUSTRY AND (AREA[InterventionType]DRUG OR AREA[InterventionType]BIOLOGICAL) AND AREA[StartDate]RANGE[2015-01-01,MAX]',
 'page_size': 50,
 'pages_downloaded': 4,
 'records_downloaded': 200,
 'development_limit': 200,
 'raw_file': 'clinicaltrials_project_dev_200.json',
 'sha256': 'ab9b8592a9e802ea4779eb93e64416c91e86c983f1f84bc3fe6d9cce56d87998'}

In [6]:
manifest_path = Path("../data/raw/clinicaltrials_project_dev_200_manifest.json")

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print("Manifest saved:", manifest_path)
print("File exists:", manifest_path.exists())

Manifest saved: ..\data\raw\clinicaltrials_project_dev_200_manifest.json
File exists: True


In [7]:
# Build trial-level table from 200-study extract

trial_rows = []

for study in all_studies:

    protocol = study.get("protocolSection", {})

    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    design = protocol.get("designModule", {})

    trial_rows.append({
        "nct_id": identification.get("nctId"),
        "brief_title": identification.get("briefTitle"),
        "overall_status": status.get("overallStatus"),
        "start_date": status.get("startDateStruct", {}).get("date"),
        "completion_date": status.get("completionDateStruct", {}).get("date"),
        "lead_sponsor": sponsor.get("leadSponsor", {}).get("name"),
        "sponsor_class": sponsor.get("leadSponsor", {}).get("class"),
        "study_type": design.get("studyType"),
        "phases": design.get("phases"),
        "enrollment": design.get("enrollmentInfo", {}).get("count")
    })

trials_df = pd.DataFrame(trial_rows)

trials_df.head()

,nct_id,brief_title,overall_status,start_date,completion_date,lead_sponsor,sponsor_class,study_type,phases,enrollment
0,NCT06695130,Study of a Combination Vaccine Comprised of Di...,COMPLETED,2024-11-18,2026-04-03,Sanofi,INDUSTRY,INTERVENTIONAL,"[PHASE1, PHASE2]",980
1,NCT05035030,Long-term Safety and Efficacy of Odevixibat in...,ACTIVE_NOT_RECRUITING,2021-09-03,2026-12-31,"Albireo, an Ipsen Company",INDUSTRY,INTERVENTIONAL,[PHASE3],62
2,NCT03754465,Clinical Study of ALLO-ASC-SHEET in Subjects w...,COMPLETED,2019-01-02,2023-10-23,"Anterogen Co., Ltd.",INDUSTRY,INTERVENTIONAL,[PHASE2],66
3,NCT03319667,A Study to Investigate the Clinical Benefit of...,ACTIVE_NOT_RECRUITING,2017-12-07,2027-06-30,Sanofi,INDUSTRY,INTERVENTIONAL,[PHASE3],475
4,NCT04475107,The Efficacy and Safety of Pyramax in Mild to ...,COMPLETED,2020-07-09,2021-04-15,Shin Poong Pharmaceutical Co. Ltd.,INDUSTRY,INTERVENTIONAL,[PHASE2],113


In [8]:
print("Rows:", len(trials_df))
print("Unique NCT IDs:", trials_df["nct_id"].nunique())
print("Duplicate NCT IDs:", trials_df["nct_id"].duplicated().sum())

trials_df.info()

Rows: 200
Unique NCT IDs: 200
Duplicate NCT IDs: 0
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   nct_id           200 non-null    str   
 1   brief_title      200 non-null    str   
 2   overall_status   200 non-null    str   
 3   start_date       200 non-null    str   
 4   completion_date  199 non-null    str   
 5   lead_sponsor     200 non-null    str   
 6   sponsor_class    200 non-null    str   
 7   study_type       200 non-null    str   
 8   phases           200 non-null    object
 9   enrollment       200 non-null    int64 
dtypes: int64(1), object(1), str(8)
memory usage: 15.8+ KB


In [9]:
# Build interventions child table from 200-study extract

intervention_rows = []

for study in all_studies:

    protocol = study.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    arms_interventions = protocol.get("armsInterventionsModule", {})

    nct_id = identification.get("nctId")
    interventions = arms_interventions.get("interventions", [])

    for intervention in interventions:
        intervention_rows.append({
            "nct_id": nct_id,
            "intervention_type": intervention.get("type"),
            "intervention_name": intervention.get("name"),
            "description": intervention.get("description")
        })

interventions_df = pd.DataFrame(intervention_rows)

interventions_df.head()

,nct_id,intervention_type,intervention_name,description
0,NCT06695130,BIOLOGICAL,RIV (recombinant influenza vaccine),"Influenza, inactivated, split virus or surface..."
1,NCT06695130,BIOLOGICAL,rC19 (dose 1),Protein subunit
2,NCT06695130,BIOLOGICAL,RIV + rC19 (dose 1),"RIV component: Influenza, inactivated, split v..."
3,NCT06695130,BIOLOGICAL,RIV + rC19 (dose 2),"RIV component: Influenza, inactivated, split v..."
4,NCT06695130,BIOLOGICAL,RIV + rC19 (dose 3),"RIV component: Influenza, inactivated, split v..."


In [10]:
print("Intervention rows:", len(interventions_df))
print("Studies represented:", interventions_df["nct_id"].nunique())

invalid_intervention_keys = interventions_df[
    ~interventions_df["nct_id"].isin(trials_df["nct_id"])
]

print("Invalid foreign keys:", len(invalid_intervention_keys))

interventions_df.info()

Intervention rows: 450
Studies represented: 200
Invalid foreign keys: 0
<class 'pandas.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   nct_id             450 non-null    str  
 1   intervention_type  450 non-null    str  
 2   intervention_name  450 non-null    str  
 3   description        436 non-null    str  
dtypes: str(4)
memory usage: 14.2 KB


In [11]:
# Build conditions child table from 200-study extract

condition_rows = []

for study in all_studies:

    protocol = study.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    conditions_module = protocol.get("conditionsModule", {})

    nct_id = identification.get("nctId")
    conditions = conditions_module.get("conditions", [])

    for condition in conditions:
        condition_rows.append({
            "nct_id": nct_id,
            "condition": condition
        })

conditions_df = pd.DataFrame(condition_rows)

conditions_df.head()

,nct_id,condition
0,NCT06695130,COVID-19 Immunization
1,NCT06695130,Influenza Immunization
2,NCT05035030,Alagille Syndrome
3,NCT03754465,Diabetic Foot Ulcer
4,NCT03319667,Plasma Cell Myeloma


In [12]:
print("Condition rows:", len(conditions_df))
print("Studies represented:", conditions_df["nct_id"].nunique())

invalid_condition_keys = conditions_df[
    ~conditions_df["nct_id"].isin(trials_df["nct_id"])
]

print("Invalid foreign keys:", len(invalid_condition_keys))

conditions_df.info()

Condition rows: 309
Studies represented: 200
Invalid foreign keys: 0
<class 'pandas.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   nct_id     309 non-null    str  
 1   condition  309 non-null    str  
dtypes: str(2)
memory usage: 5.0 KB


In [13]:
# Build locations child table from 200-study extract

location_rows = []

for study in all_studies:

    protocol = study.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    contacts_locations = protocol.get("contactsLocationsModule", {})

    nct_id = identification.get("nctId")
    locations = contacts_locations.get("locations", [])

    for location in locations:
        location_rows.append({
            "nct_id": nct_id,
            "facility": location.get("facility"),
            "city": location.get("city"),
            "state": location.get("state"),
            "country": location.get("country")
        })

locations_df = pd.DataFrame(location_rows)

locations_df.head()

,nct_id,facility,city,state,country
0,NCT06695130,Central Phoenix Medical Clinic- Site Number : ...,Phoenix,Arizona,United States
1,NCT06695130,"Synexus Clinical Research US, Inc. - Cerritos-...",Cerritos,California,United States
2,NCT06695130,Synexus Clinical Research US - Vista- Site Num...,Vista,California,United States
3,NCT06695130,Optimal Research - Florida- Site Number : 8400006,Melbourne,Florida,United States
4,NCT06695130,Synexus Clinical Research US - Orlando- Site N...,Orlando,Florida,United States


In [14]:
print("Location rows:", len(locations_df))
print("Studies represented:", locations_df["nct_id"].nunique())

invalid_location_keys = locations_df[
    ~locations_df["nct_id"].isin(trials_df["nct_id"])
]

print("Invalid foreign keys:", len(invalid_location_keys))

locations_df.info()

Location rows: 3534
Studies represented: 188
Invalid foreign keys: 0
<class 'pandas.DataFrame'>
RangeIndex: 3534 entries, 0 to 3533
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   nct_id    3534 non-null   str  
 1   facility  3516 non-null   str  
 2   city      3534 non-null   str  
 3   state     2201 non-null   str  
 4   country   3534 non-null   str  
dtypes: str(5)
memory usage: 138.2 KB


In [15]:
# Clean trial-level date fields

trials_df["start_date"] = pd.to_datetime(
    trials_df["start_date"],
    errors="coerce"
)

trials_df["completion_date"] = pd.to_datetime(
    trials_df["completion_date"],
    errors="coerce"
)

print(trials_df[["start_date", "completion_date"]].dtypes)

start_date         datetime64[us]
completion_date    datetime64[us]
dtype: object


In [16]:
# Clean trial-level date fields

trials_df["start_date"] = pd.to_datetime(
    trials_df["start_date"],
    errors="coerce",
    format="mixed"
)

trials_df["completion_date"] = pd.to_datetime(
    trials_df["completion_date"],
    errors="coerce",
    format="mixed"
)

print(trials_df[["start_date", "completion_date"]].dtypes)

start_date         datetime64[us]
completion_date    datetime64[us]
dtype: object


In [17]:
# Study type and sponsor coverage

print("Study types:")
print(trials_df["study_type"].value_counts(dropna=False))

print("\nSponsor classes:")
print(trials_df["sponsor_class"].value_counts(dropna=False))

print("\nOverall status:")
print(trials_df["overall_status"].value_counts(dropna=False))

Study types:
study_type
INTERVENTIONAL    200
Name: count, dtype: int64

Sponsor classes:
sponsor_class
INDUSTRY    200
Name: count, dtype: int64

Overall status:
overall_status
COMPLETED                  108
UNKNOWN                     24
RECRUITING                  21
TERMINATED                  16
ACTIVE_NOT_RECRUITING       15
NOT_YET_RECRUITING           9
WITHDRAWN                    5
ENROLLING_BY_INVITATION      1
SUSPENDED                    1
Name: count, dtype: int64


In [18]:
# Intervention type coverage

print(
    interventions_df["intervention_type"]
    .value_counts(dropna=False)
)
# Date coverage

print("Earliest start:", trials_df["start_date"].min())
print("Latest start:", trials_df["start_date"].max())

intervention_type
DRUG                   370
BIOLOGICAL              58
OTHER                   12
DEVICE                   7
PROCEDURE                2
COMBINATION_PRODUCT      1
Name: count, dtype: int64
Earliest start: 2015-01-07 00:00:00
Latest start: 2026-12-01 00:00:00


In [19]:
# Final structural assertions

assert len(trials_df) == 200
assert trials_df["nct_id"].is_unique
assert trials_df["nct_id"].notna().all()

assert interventions_df["nct_id"].isin(trials_df["nct_id"]).all()
assert conditions_df["nct_id"].isin(trials_df["nct_id"]).all()
assert locations_df["nct_id"].isin(trials_df["nct_id"]).all()

print("All structural QA checks passed.")

All structural QA checks passed.


In [20]:
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

In [21]:
# Save processed analytical tables

processed_path = Path("../data/processed")

trials_df.to_csv(
    processed_path / "trials_dev_200.csv",
    index=False
)

interventions_df.to_csv(
    processed_path / "interventions_dev_200.csv",
    index=False
)

conditions_df.to_csv(
    processed_path / "conditions_dev_200.csv",
    index=False
)

locations_df.to_csv(
    processed_path / "locations_dev_200.csv",
    index=False
)

print("Processed tables saved.")

Processed tables saved.


In [22]:
for file in processed_path.glob("*_dev_200.csv"):
    print(file.name, "-", file.exists())

conditions_dev_200.csv - True
interventions_dev_200.csv - True
locations_dev_200.csv - True
trials_dev_200.csv - True


In [23]:
# Day 10 final validation summary

print("=== DAY 10 INGESTION SUMMARY ===")

print("\nRaw extraction:")
print("Studies downloaded:", len(all_studies))
print("Pages downloaded:", pages_downloaded)

print("\nTrial table:")
print("Rows:", len(trials_df))
print("Unique NCT IDs:", trials_df["nct_id"].nunique())
print("Duplicate NCT IDs:", trials_df["nct_id"].duplicated().sum())

print("\nChild tables:")
print("Intervention rows:", len(interventions_df))
print("Condition rows:", len(conditions_df))
print("Location rows:", len(locations_df))

print("\nForeign key validation:")
print(
    "Interventions:",
    interventions_df["nct_id"].isin(trials_df["nct_id"]).all()
)
print(
    "Conditions:",
    conditions_df["nct_id"].isin(trials_df["nct_id"]).all()
)
print(
    "Locations:",
    locations_df["nct_id"].isin(trials_df["nct_id"]).all()
)

print("\nDAY 10 PIPELINE COMPLETE")

=== DAY 10 INGESTION SUMMARY ===

Raw extraction:
Studies downloaded: 200
Pages downloaded: 4

Trial table:
Rows: 200
Unique NCT IDs: 200
Duplicate NCT IDs: 0

Child tables:
Intervention rows: 450
Condition rows: 309
Location rows: 3534

Foreign key validation:
Interventions: True
Conditions: True
Locations: True

DAY 10 PIPELINE COMPLETE
